# Praktikum 08 – Hardware und Inferenz-Optimierung
**Applied Generative AI – NLP | Sommersemester 2026**

In diesem Notebook untersuchen wir die Hardware-Anforderungen für Large Language Models und lernen Techniken zur Optimierung der Inferenz kennen. Wir berechnen Speicherbedarfe, erklären den KV-Cache, führen Quantisierungen durch und betrachten moderne Serving-Frameworks.

**Lernziele:**
- GPU-Markt 2026 verstehen und Speicherbedarf berechnen
- KV-Cache erklären und dessen Größe bestimmen
- Quantisierung von FP32 bis Q4 durchführen
- GGUF-Quantisierung mit llama.cpp anwenden
- Serving-Optimierungen wie FlashAttention und vLLM kennenlernen

**Zeitbedarf:** ca. 90 Minuten

In [ ]:
import importlib
import os
import shutil
import subprocess
import sys
import time
from importlib.util import find_spec

IN_COLAB = "google.colab" in sys.modules
MODEL = os.getenv("LLM_MODEL", "qwen2.5:0.5b").strip()
OLLAMA_BASE_URL = os.getenv("OLLAMA_BASE_URL", "http://127.0.0.1:11434").strip()

REQUIRED = {
    "numpy": ("numpy", "2.4.4"),
    "requests": ("requests", "2.33.1"),
}


def run_install(specs):
    if shutil.which("uv") is None:
        raise RuntimeError("uv is not installed. Install uv and rerun the setup cell.")

    command = ["uv", "pip", "install", "--python", sys.executable, *specs]
    in_venv = sys.prefix != getattr(sys, "base_prefix", sys.prefix) or bool(os.environ.get("VIRTUAL_ENV"))
    if not in_venv:
        command.append("--system")

    print("$", " ".join(command))
    subprocess.check_call(command)


missing_specs = [
    f"{install_name}=={version}"
    for import_name, (install_name, version) in REQUIRED.items()
    if find_spec(import_name) is None
]
if missing_specs:
    run_install(missing_specs)
    importlib.invalidate_caches()

missing_after = [import_name for import_name in REQUIRED if find_spec(import_name) is None]
if missing_after:
        raise RuntimeError(f"Missing packages after installation: {', '.join(missing_after)}")

import numpy as np

print("✓ Setup abgeschlossen")
print(f"  Modell: {MODEL}")
print(f"  Ollama: {OLLAMA_BASE_URL}")

---

## Teil 1: GPU-Markt 2026 und Speicherberechnung

--- 

## Teil 1: GPU-Markt 2026 und Speicherberechnung

Der GPU-Markt hat sich durch die steigende Nachfrage nach KI-Anwendungen dramatisch verändert. Für das Deployment von LLMs sind insbesondere die VRAM-Kapazität und die Rechenleistung relevant.

### Wichtige GPU-Kategorien 2026:

| Kategorie | VRAM | Typische GPUs | Einsatz |
|-----------|------|---------------|--------|
| Consumer | 8-24 GB | RTX 4060 Ti - RTX 5090 | Entwicklung, kleine Modelle |
| Professional | 24-80 GB | A5000 - A100, H100 | Produktion, mittlere Modelle |
| High-End | 80-192 GB | H100 (80GB), H200 (141GB) | Große Modelle, Forschung |

### Grundlegende Speicherformel

Die Basisformel für den Modell-Speicherbedarf lautet:

**Speicherbedarf (GB) = Anzahl Parameter × Bytes pro Parameter**

Beispiel: 1 Milliarde Parameter × 2 Bytes (FP16) = 2 GB VRAM

In [ ]:
# Speicherberechnung für verschiedene Modellgrößen und Präzisionen

def berechne_speicher(parameter_milliarden, praezision):
    """
    Berechnet den Speicherbedarf in GB.
    
    Args:
        parameter_milliarden: Anzahl der Parameter in Milliarden
        praezision: Präzision als String ('fp32', 'fp16', 'bf16', 'int8', 'q4_0')
    
    Returns:
        Speicherbedarf in GB (nur Modellgewichte, ohne KV-Cache, Optimizer etc.)
    """
    # Bytes pro Parameter für verschiedene Präzisionen
    bytes_pro_parameter = {
        'fp32': 4.0,   # Full Precision: 32 Bit
        'fp16': 2.0,   # Half Precision: 16 Bit
        'bf16': 2.0,   # Brain Float: 16 Bit
        'int8': 1.0,   # Integer 8 Bit
        'q4_0': 0.5,   # Quantisiert Q4: ca. 4 Bit
        'q4_k': 0.4,   # Q4 mit特殊 Block-Kompression
        'q2_k': 0.25,  # Q2: ca. 2 Bit
    }
    
    if praezision not in bytes_pro_parameter:
        raise ValueError(f"Unbekannte Präzision: {praezision}")
    
    bytes_pro_param = bytes_pro_parameter[praezision]
    parameter = parameter_milliarden * 1e9  # Umrechnung in absolute Zahl
    speicher_gb = (parameter * bytes_pro_param) / 1e9
    
    return speicher_gb


# Vergleich verschiedener Modellgrößen
modell_groessen = [0.5, 1, 3, 7, 70]  # in Milliarden Parameter
prazisionen = ['fp32', 'fp16', 'bf16', 'int8', 'q4_0']

print("=" * 70)
print("Speicherbedarf verschiedener Modellgrößen (in GB)")
print("=" * 70)
print(f"{'Modellgröße':<15} {'FP32':>10} {'FP16/BF16':>10} {'INT8':>10} {'Q4':>10}")
print("-" * 70)

for groesse in modell_groessen:
    fp32 = berechne_speicher(groesse, 'fp32')
    fp16 = berechne_speicher(groesse, 'fp16')
    int8 = berechne_speicher(groesse, 'int8')
    q4 = berechne_speicher(groesse, 'q4_0')
    print(f"{groesse}B Parameter   {fp32:>10.1f} {fp16:>10.1f} {int8:>10.1f} {q4:>10.1f}")

print("-" * 70)
print("\nHinweis: Diese Werte gelten nur für die Modellgewichte!")
print("Zusätzlich benötigen Sie: KV-Cache, Aktivierungen, Optimizer-State (Training).")

### Praktische GPU-Empfehlungen

Basierend auf den Speicherberechnungen:

| Modellgröße | Minimale GPU | Empfohlene GPU |
|-------------|--------------|----------------|
| 0.5B Q4 | 2 GB VRAM | RTX 3060 (12GB) |
| 1B FP16 | 4 GB VRAM | RTX 4060 Ti (8GB) |
| 3B FP16 | 8 GB VRAM | RTX 4070 (12GB) |
| 7B FP16 | 16 GB VRAM | RTX 4080 (16GB) |
| 7B Q4 | 6 GB VRAM | RTX 3060 Ti (8GB) |
| 70B Q4 | 40 GB VRAM | A100 (40GB) |

**Wichtige Erkenntnis:** Durch Quantisierung können Modelle auf deutlich günstigerer Hardware laufen!

---

## Teil 2: KV-Cache – Erklärung und Berechnung

### Was ist der KV-Cache?

Bei der autoregressiven Generierung von Text muss das Modell für jedes neue Token den gesamten bisherigen Kontext berücksichtigen. Der **Key-Value-Cache (KV-Cache)** speichert die bereits berechneten Key- und Value-Attention-Matrizen, um redundante Berechnungen zu vermeiden.

### Warum ist der KV-Cache wichtig?

1. **Ohne KV-Cache:** Für jedes neue Token werden alle vorherigen Attention-Scores neu berechnet → O(n²) Komplexität
2. **Mit KV-Cache:** Nur das neue Token wird berechnet → O(n) Komplexität

### Speicherformel für KV-Cache

KV-Cache Größe = 2 × Batch_Size × Sequenzlänge × Layer × Kopfzahl × Dimension_pro_Kopf × Bytes_pro_Parameter

Vereinfacht: **KV-Cache ≈ 2 × Batch × Seq_Len ×hidden_dim × bytes_per_param**

In [ ]:
def berechne_kv_cache(
    hidden_dim=2048,       # Hidden Dimension (z.B. 2048 für 2B Modell)
    num_layers=24,         # Anzahl Transformer-Layer
    num_heads=16,          # Anzahl Attention-Heads
    head_dim=128,          # Dimension pro Head (hidden_dim / num_heads)
    seq_len=4096,          # Maximale Sequenzlänge
    batch_size=1,          # Batch-Größe
    praezision='fp16'     # Präzision
):
    """
    Berechnet den KV-Cache Speicherbedarf.
    
    Der KV-Cache speichert K und V Matrizen für jeden Layer.
    Jede Matrix hat die Form: [batch, seq_len, num_heads, head_dim]
    """
    bytes_pro_parameter = {
        'fp32': 4.0,
        'fp16': 2.0,
        'bf16': 2.0,
        'int8': 1.0,
        'q4_0': 0.5,
    }
    
    bytes_param = bytes_pro_parameter.get(praezision, 2.0)
    
    # KV-Cache hat 2 Komponenten (Keys + Values) pro Layer
    # Größe = 2 * batch * seq_len * num_layers * num_heads * head_dim * bytes
    cache_einschichtig = batch_size * seq_len * num_heads * head_dim * bytes_param
    cache gesamt = 2 * num_layers * cache_einschichtig  # K + V
    
    # Umrechnung in MB/GB
    cache_mb = cache_gesamt / (1024 ** 2)
    cache_gb = cache_gesamt / (1024 ** 3)
    
    return cache_mb, cache_gb


# Beispiel: 7B Modell (LLaMA-7B ähnlich)
print("=" * 60)
print("KV-Cache Berechnung für verschiedene Szenarien")
print("=" * 60)

# Typische 7B Modell-Parameter (LLaMA-7B style)
config_7b = {
    'hidden_dim': 4096,
    'num_layers': 32,
    'num_heads': 32,
    'head_dim': 128,
}

# Verschiedene Sequenzlängen und Batch-Größen
szenarien = [
    (512, 1, "Kurze Queries"),
    (2048, 1, "Längere Dokumente"),
    (4096, 1, "Maximale Context"),
    (4096, 4, "Batch-Verarbeitung"),
]

print(f"\n7B Modell (hidden={config_7b['hidden_dim']}, layers={config_7b['num_layers']})")
print("-" * 60)
print(f"{'Szenario':<25} {'Seq Len':>10} {'Batch':>8} {'KV-Cache':>12}")
print("-" * 60)

for seq_len, batch, name in szenarien:
    cache_mb, _ = berechne_kv_cache(
        hidden_dim=config_7b['hidden_dim'],
        num_layers=config_7b['num_layers'],
        num_heads=config_7b['num_heads'],
        head_dim=config_7b['head_dim'],
        seq_len=seq_len,
        batch_size=batch
    )
    print(f"{name:<25} {seq_len:>10} {batch:>8} {cache_mb:>10.1f} MB")

print("-" * 60)
print("\n→ Bei kurzen Sequenzen ist der KV-Cache vernachlässigbar.")
print("→ Bei langen Contexts (4096+) wird der KV-Cache zum Flaschenhals!")

### KV-Cache als Flaschenhals

Bei langen Kontexten kann der KV-Cache den gesamten verfügbaren VRAM belegen:

- **LLaMA-2 7B @ 32k Context:** ~12 GB KV-Cache (FP16)
- **KV-Cache Quantisierung:** Kann den Speicher um 50-75% reduzieren

Moderne Optimierungen wie **PagedAttention** (vLLM) reduzieren die Fragmentierung des KV-Caches.

---

## Teil 3: Quantisierung – Von FP32 bis Q4

### Was ist Quantisierung?

Quantisierung reduziert die Präzision der Modellgewichte, um Speicher und Rechenaufwand zu verringern. Der Grundgedanke: Nicht jede Zahl braucht 32 Bit Genauigkeit.

### Präzisionsstufen im Überblick:

| Format | Bits | Bytes/Param | Verwendung |
|--------|------|-------------|------------|
| FP32 | 32 | 4 | Training, genaue Berechnungen |
| FP16 | 16 | 2 | Standard-Inferenz |
| BF16 | 16 | 2 | Training, bessere numerische Stabilität |
| INT8 | 8 | 1 | Quantisierte Inferenz |
| Q4_0 | ~4 | 0.5 | Stark komprimiert, kaum Qualitätsverlust |
| Q2_K | ~2 | 0.25 | Extreme Komprimierung, für Tests |

### Wie funktioniert Quantisierung?

Bei der einfachen **Dynamic Quantization** werden die Gewichte auf den Bereich [-127, 127] (INT8) abgebildet:

```
quantized = round(float_value / scale)
float_value = quantized * scale
```

Die **Scale** wird aus dem Maximalwert der Gewichte berechnet.

In [ ]:
# Demonstration der Quantisierung mit Python
import numpy as np

np.random.seed(42)

# Simulierte Gewichte eines Layers (z.B. 1000 Werte)
gewichter = np.random.randn(1000).astype(np.float32) * 2.0

print("=" * 60)
print("Quantisierungs-Demonstration")
print("=" * 60)

# FP32 (Original)
print(f"\nFP32 (Original):")
print(f"  Wertebereich: [{gewichter.min():.4f}, {gewichter.max():.4f}]")
print(f"  Speicher: {gewichter.nbytes / 1024:.1f} KB")

# FP16 Quantisierung
gewichter_fp16 = gewichter.astype(np.float16)
print(f"\nFP16:")
print(f"  Wertebereich: [{gewichter_fp16.min():.4f}, {gewichter_fp16.max():.4f}]")
print(f"  Speicher: {gewichter_fp16.nbytes / 1024:.1f} KB")

# INT8 Quantisierung (Symmetrisch)
max_val = np.abs(gewichter).max()
scale_int8 = max_val / 127.0
gewichter_int8 = np.round(gewichter / scale_int8).astype(np.int8)
gewichter_int8_dequant = gewichter_int8.astype(np.float32) * scale_int8

print(f"\nINT8 (Quantisiert):")
print(f"  Scale: {scale_int8:.6f}")
print(f"  Speicher: {gewichter_int8.nbytes / 1024:.1f} KB")

# Fehlerberechnung
fehler_fp16 = np.mean(np.abs(gewichter - gewichter_fp16.astype(np.float32)))
fehler_int8 = np.mean(np.abs(gewichter - gewichter_int8_dequant))

print(f"\nDurchschnittlicher Absolutfehler:")
print(f"  FP16: {fehler_fp16:.6f}")
print(f"  INT8: {fehler_int8:.6f}")

# Q4 Simulation (vereinfacht)
scale_q4 = max_val / 7.0  # 4-Bit = -7 bis +7
gewichter_q4 = np.round(gewichter / scale_q4).astype(np.int8)
gewichter_q4 = np.clip(gewichter_q4, -7, 7)  # Clipping
gewichter_q4_dequant = gewichter_q4.astype(np.float32) * scale_q4

fehler_q4 = np.mean(np.abs(gewichter - gewichter_q4_dequant))
print(f"  Q4 (simuliert): {fehler_q4:.6f}")

print("\n→ Höhere Komprimierung = höherer Fehler = potenziell schlechtere Qualität")

### GGUF-Quantisierungsstufen (llama.cpp)

Das GGUF-Format von llama.cpp bietet verschiedene Stufen:

| Stufe | Bits | Qualität | Geschwindigkeit | Empfehlung |
|-------|------|----------|-----------------|------------|
| Q8_0 | 8 | ~99% | Schnell | Falls möglich |
| Q6_K | 6 | ~98% | Sehr gut | Beste Balance |
| Q5_K | 5 | ~97% | Gut | Häufig verwendet |
| Q4_K | 4 | ~96% | Gut | Sehr beliebt |
| Q4_0 | 4 | ~95% | Gut | Einfache Variante |
| Q3_K | 3 | ~94% | Akzeptabel | Für wenig VRAM |
| Q2_K | 2 | ~90% | Mittel | Notfall |

**K = Block-Quantisierung** mit spezieller Skalierung pro Block (meist bessere Qualität)

---

## Teil 4: GGUF-Quantisierung mit llama.cpp

In diesem Abschnitt führen wir eine echte Quantisierung mit llama.cpp durch. Wir werden:
1. llama.cpp installieren (falls nicht vorhanden)
2. Ein Modell herunterladen
3. Das Modell in verschiedene GGUF-Formate quantisieren
4. Die resultierenden Dateigrößen vergleichen

In [ ]:
# Setup: Prüfe verfügbare Tools und installiere falls nötig
import os
import shutil
import subprocess
import sys

IN_COLAB = "google.colab" in sys.modules
WORK_DIR = "/tmp/llama_quantization" if IN_COLAB else "./llama_quantization"
MODEL_NAME = os.getenv("LLM_MODEL", "qwen2.5:0.5b").strip()
OLLAMA_BASE_URL = os.getenv("OLLAMA_BASE_URL", "http://127.0.0.1:11434").strip()

# Erstelle Arbeitsverzeichnis
os.makedirs(WORK_DIR, exist_ok=True)
os.chdir(WORK_DIR)

print(f"Arbeitsverzeichnis: {WORK_DIR}")
print(f"Ziel-Modell: {MODEL_NAME}")

# Prüfe ob Ollama verfügbar ist
def check_ollama():
    try:
        import requests
        response = requests.get(f"{OLLAMA_BASE_URL}/api/tags", timeout=5)
        response.raise_for_status()
        return True
    except Exception:
        return False

def is_local_host(url):
    host = url.split("://", 1)[-1].split("/", 1)[0].split(":", 1)[0].lower()
    return host in {"127.0.0.1", "localhost", "0.0.0.0"}

def wait_for_ollama(base_url, timeout=60):
    import time
    import requests
    deadline = time.time() + timeout
    while time.time() < deadline:
        try:
            response = requests.get(f"{base_url.rstrip('/')}/api/tags", timeout=5)
            response.raise_for_status()
            return True
        except Exception:
            time.sleep(2)
    return False

def install_ollama():
    """Installiert Ollama falls nicht vorhanden."""
    if shutil.which("ollama"):
        return True
    
    print("Ollama wird installiert...")
    
    if IN_COLAB:
        # Colab: Ollama manuell installieren
        subprocess.run(["pip", "install", "-q", "zstandard"], check=True)
        result = subprocess.run(
            ["sh", "-c", "curl -fsSL https://ollama.com/install.sh | sh"],
            capture_output=True, text=True, timeout=120
        )
        if result.returncode != 0:
            print(f"Ollama Installation fehlgeschlagen: {result.stderr}")
            return False
    else:
        # Lokal: Standard-Installation
        print("Bitte installieren Sie Ollama manuell: https://ollama.com")
        return False
    
    return shutil.which("ollama") is not None

# Ollama-Verfügbarkeit prüfen
if not is_local_host(OLLAMA_BASE_URL):
    print(f"\nWarnung: {OLLAMA_BASE_URL} ist kein lokaler Host.")
    print("Für diese Übung wird ein lokaler Ollama-Server benötigt.")
    print("Bitte starten Sie 'ollama serve' und führen Sie diese Zelle erneut aus.")
    raise RuntimeError("Lokaler Ollama-Server erforderlich")

if not check_ollama():
    if not install_ollama():
        raise RuntimeError("Ollama konnte nicht installiert werden")
    if not wait_for_ollama(OLLAMA_BASE_URL, timeout=60):
        raise RuntimeError("Ollama-Server antwortet nicht")

print("✓ Ollama ist verfügbar")

In [ ]:
# Exportiere das Modell nach GGUF mit ollama
import subprocess

# Extrahiere Modell-Name ohne Tag
base_model = MODEL_NAME.split(":")[0]

print(f"Exportiere Modell '{MODEL_NAME}' nach GGUF...")
print(f"Dies kann je nach Modellgröße einige Minuten dauern.")

gguf_path = os.path.join(WORK_DIR, f"{base_model}.gguf")

# Exportiere mit ollama
result = subprocess.run(
    ["ollama", "show", MODEL_NAME, "--gguf"],
    capture_output=True,
    text=True,
    cwd=WORK_DIR
)

if result.returncode != 0:
    print(f"Export fehlgeschlagen: {result.stderr}")
    print("\nVersuche Alternative: Direkter Export via ollama...")
    
    # Alternative: Nutze die Modell-Dateien direkt
    # ollama speichert Modelle unter ~/.ollama/models/
    import os.path
    
    ollama_models = os.path.expanduser("~/.ollama/models")
    if os.path.exists(ollama_models):
        print(f"Ollama Models-Verzeichnis: {ollama_models}")
        for root, dirs, files in os.walk(ollama_models):
            for f in files:
                if f.endswith('.gguf'):
                    print(f"  Gefunden: {os.path.join(root, f)}")
    
    raise RuntimeError("GGUF-Export nicht verfügbar. Bitte nutzen Sie eine alternative Methode.")

print(result.stdout)
print("✓ Modell exportiert")

### Alternative: Direkter Vergleich ohne Quantisierung

Da der direkte GGUF-Export in dieser Umgebung möglicherweise nicht verfügbar ist, zeigen wir hier einen alternativen Ansatz mit Python-Bibliotheken.

In [ ]:
# Alternative: Demonstration der Quantisierung mit Python
# Wir simulieren die verschiedenen GGUF-Stufen

import numpy as np

print("=" * 70)
print("Quantisierungs-Vergleich: Verschiedene GGUF-Stufen")
print("=" * 70)

# Lade ein Beispiel-Modell (zufällige Gewichte simuliert)
# In der Praxis würden Sie echte Modellgewichte verwenden
np.random.seed(123)

# Simuliere Gewichte für verschiedene Modellgrößen
modell_groessen = {
    "500M Parameter": 500_000_000,
    "1B Parameter": 1_000_000_000,
    "3B Parameter": 3_000_000_000,
    "7B Parameter": 7_000_000_000,
}

quantisierungs_stufen = {
    "FP32 (Original)": 4.0,
    "FP16": 2.0,
    "Q8_0": 1.0,
    "Q6_K": 0.75,
    "Q5_K": 0.625,
    "Q4_K": 0.5,
    "Q4_0": 0.5,
    "Q3_K": 0.4375,
    "Q2_K": 0.3125,
}

print(f"\n{'Modellgröße':<20}", end="")
for stufe in quantisierungs_stufen:
    print(f"{stufe:>12}", end="")
print()
print("-" * 120)

for name, params in modell_groessen.items():
    print(f"{name:<20}", end="")
    for stufe, bytes_pro_param in quantisierungs_stufen.items():
        speicher_gb = (params * bytes_pro_param) / 1e9
        print(f"{speicher_gb:>11.2f}G", end="")
    print()

print("-" * 120)
print("\nErkenntnis: Ein 7B Modell benötigt in Q4_0 nur ~3.5 GB statt ~14 GB!")
print("Das macht es möglich, große Modelle auf Consumer-GPUs zu betreiben.")

In [ ]:
# Praktisches Beispiel: Reale Dateigrößen
print("=" * 70)
print("Praktisches Beispiel: Tatsächliche Dateigrößen")
print("=" * 70)

def format_size(bytes_size):
    """Formatiert Bytes in lesbare Größe."""
    for unit in ['B', 'KB', 'MB', 'GB']:
        if bytes_size < 1024:
            return f"{bytes_size:.1f} {unit}"
        bytes_size /= 1024
    return f"{bytes_size:.1f} TB"

# Berechne theoretische Dateigrößen für 7B Modell
params_7b = 7_000_000_000

stufen = [
    ("FP32 (Original)", 4.0),
    ("FP16", 2.0),
    ("Q8_0", 1.0),
    ("Q6_K", 0.75),
    ("Q5_K_M", 0.625),
    ("Q5_K_S", 0.5625),
    ("Q4_K_M", 0.5),
    ("Q4_0", 0.5),
    ("Q3_K_M", 0.4375),
    ("Q2_K", 0.3125),
]

print(f"\n7B Modell (~{format_size(params_7b * 4)} als FP32):\n")
print(f"{'Stufe':<15} {'Bytes/Param':>12} {'Dateigröße':>15} {'Kompression':>12}")
print("-" * 60)

original_size = params_7b * 4.0
for stufe, bytes_p in stufen:
    size = params_7b * bytes_p
    compression = (1 - size / original_size) * 100
    print(f"{stufe:<15} {bytes_p:>12.4f} {format_size(size):>15} {compression:>11.1f}%")

print("-" * 60)
print("\n✓ Q4_0 erreicht ~87.5% Kompression bei akzeptablem Qualitätsverlust")

---

## Teil 5: Serving-Optimierungen

### FlashAttention

FlashAttention ist ein Algorithmus, der die Attention-Berechnung optimiert durch:

1. **IO-Awareness:** Minimiert Speicherzugriffe zwischen HBM und SRAM
2. **Tiling:** Berechnet Attention in Blöcken statt gesamter Matrix
3. **Kein vollständiger KV-Cache im VRAM:** Reduziert Speicher von O(N²) auf O(N)

**Vorteile:**
- 2-4x schneller als Standard-Attention
- Bis zu 20x weniger VRAM für denselben Context
- Verbesserte numerische Stabilität

**Nachteile:**
- Nur auf unterstützter Hardware (NVIDIA Ampere+)
- Kann Genauigkeit leicht beeinflussen

In [ ]:
# FlashAttention-Verfügbarkeit prüfen
print("=" * 60)
print("FlashAttention Verfügbarkeitscheck")
print("=" * 60)

try:
    import torch
    print(f"PyTorch Version: {torch.__version__}")
    print(f"CUDA verfügbar: {torch.cuda.is_available()}")
    
    if torch.cuda.is_available():
        print(f"GPU: {torch.cuda.get_device_name(0)}")
        compute_capability = torch.cuda.get_device_capability()
        print(f"Compute Capability: {compute_capability[0]}.{compute_capability[1]}")
        
        # Prüfe FlashAttention-Unterstützung
        if compute_capability[0] >= 8:  # Ampere oder neuer
            print("\n✓ GPU unterstützt FlashAttention (Ampere+)")
        else:
            print("\n⚠ GPU unterstützt kein FlashAttention (vor Ampere)")
            print("   FlashAttention 2 funktioniert teilweise auf Turing (RTX 20xx)")
    else:
        print("\n⚠ Keine CUDA-GPU verfügbar")
        print("   FlashAttention kann nur auf GPU genutzt werden")
        
except ImportError:
    print("PyTorch nicht installiert")
    print("FlashAttention erfordert PyTorch mit CUDA-Unterstützung")

### vLLM – High-Throughput Serving

vLLM ist ein optimiertes Serving-Framework mit PagedAttention:

**PagedAttention (Verringert KV-Cache-Fragmentierung):**
- Traditionell:连续er Speicherblock (verschwendet Platz bei variablem Context)
- Paged: Flexible, nicht-kontinuierliche Seiten (ähnlich virtueller Speicher)
- Bis zu 2.4x höherer Durchsatz als Hugging Face

**Weitere Optimierungen in vLLM:**
- Continous Batching (dynamische Batch-Größen)
- Speculative Decoding (vorsichtige Token-Vorhersage)
- Chunked Prefills (aufgeteilte Kontextverarbeitung)

In [ ]:
# vLLM-Verfügbarkeit prüfen
print("=" * 60)
print("vLLM Verfügbarkeitscheck")
print("=" * 60)

try:
    import vllm
    print(f"✓ vLLM installiert: Version {vllm.__version__}")
    print("\nvLLM kann für High-Throughput-Inferenz verwendet werden.")
    print("Beispiel-Usage:")
    print("""
from vllm import LLM, SamplingParams

# Modell laden
llm = LLM(model="meta-llama/Llama-2-7b-hf", tensor_parallel_size=1)

# Inferenz
sampling_params = SamplingParams(temperature=0.8, max_tokens=100)
outputs = llm.generate(["Erkläre Quantisierung in 2 Sätzen."], sampling_params)
print(outputs[0].outputs[0].text)
""")
    
except ImportError:
    print("✗ vLLM nicht installiert")
    print("\nInstallation:")
    print("  pip install vllm")
    print("\noder mit uv:")
    print("  uv pip install vllm")
    print("\nHinweis: vLLM erfordert CUDA 12.1+ und kompatible GPU")

### Vergleich der Serving-Frameworks

| Framework | Geschwindigkeit | VRAM-Effizienz | Features | Hardware |
|-----------|-----------------|----------------|----------|----------|
| Hugging Face | Standard | Mittel | Flexibel | Alle |
| vLLM | Sehr hoch | Sehr hoch | PagedAttention | NVIDIA Ampere+ |
| Text Generation Inference (TGI) | Hoch | Hoch | Cont. Batching | Alle |
| llama.cpp (GGUF) | Mittel | Sehr hoch | Quantisierung | CPU+GPU |

### Empfehlungen nach Anwendungsfall:

- **Entwicklung/Prototyping:** Hugging Face Transformers
- **Hoher Durchsatz:** vLLM
- **Wenig VRAM:** llama.cpp mit Q4/Q5 Quantisierung
- **CPU-Inferenz:** llama.cpp (GGUF)
- **Produktion mit Monitoring:** TGI (Hugging Face)

In [ ]:
# Zusammenfassung: Empfehlungen für verschiedene Setups
print("=" * 70)
print("Hardware-Empfehlungen nach Anwendungsfall")
print("=" * 70)

empfehlungen = [
    {
        "fall": "Laptop ohne GPU",
        "modell": "1-3B Q4-Q5",
        "framework": "llama.cpp (GGUF)",
        "ram": "8-16 GB System RAM",
    },
    {
        "fall": "Consumer-GPU (8GB)",
        "modell": "3-7B Q4-Q5",
        "framework": "llama.cpp oder HF + FlashAttention",
        "ram": "8 GB VRAM",
    },
    {
        "fall": "Consumer-GPU (12GB)",
        "modell": "7B FP16 oder 13B Q4",
        "framework": "vLLM oder HF",
        "ram": "12 GB VRAM",
    },
    {
        "fall": "Profi-GPU (24GB)",
        "modell": "13-34B Q4-Q5",
        "framework": "vLLM",
        "ram": "24 GB VRAM",
    },
    {
        "fall": "Server (A100/H100)",
        "modell": "70B+ (je nach VRAM)",
        "framework": "vLLM mit TP/PP",
        "ram": "80+ GB VRAM",
    },
]

print(f"\n{'Anwendungsfall':<25} {'Modell':<20} {'Framework':<25} {'RAM':<15}")
print("-" * 90)

for e in empfehlungen:
    print(f"{e['fall']:<25} {e['modell']:<20} {e['framework']:<25} {e['ram']:<15}")

print("-" * 90)
print("\n✓ Mit diesen Empfehlungen können Sie das richtige Setup wählen!")

---

## Zusammenfassung

In diesem Notebook haben wir folgende Themen behandelt:

### Teil 1: GPU-Markt und Speicherberechnung
- Grundformel: Speicherbedarf = Parameter × Bytes/Parameter
- 7B Modell in FP16 ≈ 14 GB, in Q4 ≈ 3.5 GB
- Consumer-GPUs reichen für quantisierte Modelle

### Teil 2: KV-Cache
- Speichert Key-Value-Attention-Matrizen
- Reduziert Rechenkomplexität von O(n²) auf O(n)
- Kann bei langen Contexts zum Flaschenhals werden

### Teil 3: Quantisierung
- FP32 → FP16 → INT8 → Q4: Immer weniger Präzision
- Q4 erreicht ~87.5% Kompression
- Qualitätsverlust meist vernachlässigbar

### Teil 4: GGUF-Quantisierung
- llama.cpp-Format mit verschiedenen Stufen
- Q4_K/Q5_K bieten beste Balance
- Ermöglicht große Modelle auf kleinen GPUs

### Teil 5: Serving-Optimierungen
- FlashAttention: IO-optimierte Attention
- vLLM: PagedAttention für hohen Durchsatz
- Wahl des Frameworks je nach Hardware und Anforderung

---

**Nächste Schritte:**
- Experimentieren Sie mit verschiedenen Quantisierungsstufen
- Testen Sie verschiedene Serving-Frameworks
- Vergleichen Sie Inferenzgeschwindigkeiten

**Hinweis:** Dieses Notebook behandelt Hardware und Inferenz-Optimierung. Agenten und Reasoning werden in P09 behandelt.